# Consecutive Date Streaks by State

## Table

**`superstore_orderstasks`** — a list of dates and their corresponding order status.

| Column | Type |
| --- | --- |
| `date_value` | date |
| `state` | varchar(10) |

## Question

Find the **start date and end date of each consecutive-day streak** for each `state`. Consecutive dates (no gaps) belong to the same streak; when a date is skipped, a new streak begins.

Return `state`, `start_date`, and `end_date`, ordered by `start_date`.

### Approach (Gaps & Islands)
1. Rank rows within each `state` by `date_value` using `row_number()`.
2. Subtract the rank (in days) from `date_value` — rows in the same consecutive streak share the same resulting `diff` date.
3. Group by `state` and `diff`; take `min(date_value)` as start and `max(date_value)` as end.

In [0]:
%sql
-- ============================================================
-- Cell 1: Create and populate the `superstore_orderstasks` table
-- ============================================================
create table b_sql.b_practice.superstore_orderstasks (
date_value date,
state varchar(10)
);

insert into b_sql.b_practice.superstore_orderstasks  values ('2019-01-01','success'),('2019-01-02','success'),('2019-01-03','success'),('2019-01-04','fail')
,('2019-01-05','fail'),('2019-01-06','success')

In [0]:
%sql
-- ============================================================
-- Cell 2: Preview the superstore_orderstasks table
-- ============================================================
select * from b_sql.b_practice.superstore_orderstasks

In [0]:
%sql
-- ============================================================
-- Cell 3: SQL solution — consecutive date streaks per state
-- CTE: row_number() per state → subtract rank from date_value
-- to get a shared diff per streak (gaps & islands)
-- Final: group by state + diff → min/max date_value
-- ============================================================
with cte as(select *,row_number() over(partition  by state order by date_value) as rnk, dateadd(day,-1*row_number() over(partition  by state order by date_value),date_value) as new_date from b_sql.b_practice.superstore_orderstasks)
select min(date_value)as start_date,max(date_value)as end_date,state from cte group by state,new_date order by start_date

In [0]:
# ============================================================
# Cell 4: Import PySpark SQL functions, types, and Window
# (Window is needed for row_number() in the solution)
# ============================================================
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# ============================================================
# Cell 5: Load superstore_orderstasks into a PySpark DataFrame
# ============================================================
df_task=spark.read.table("b_sql.b_practice.superstore_orderstasks")
df_task.display()

In [0]:
# ============================================================
# Cell 6: PySpark solution — consecutive date streaks per state
# Step 1: row_number() per state ordered by date_value
# Step 2: dateadd(date_value, -rnk) → shared diff per streak
# Step 3: groupBy state + diff → min/max date_value
# ============================================================
df_diff=df_task.withColumn("rnk", row_number().over(Window.partitionBy("state").orderBy(col("date_value")))).withColumn("diff",dateadd(col("date_value"),-col("rnk")))
df_diff.display()

df_final=df_diff.groupBy("state","diff").agg(min("date_value").alias("start_date"),max("date_value").alias("end_date")).orderBy(col("start_date"))
df_final.drop("diff").display()